In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2

from torchvision import datasets
from torch.utils.data import DataLoader

import timm
from timm.data import resolve_data_config, create_transform

from PIL import Image

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


# =========================================================
# CONFIGURATION
# =========================================================
class CFG:
    test_dir = "./data/Test"
    model_name = "swin_small_patch4_window7_224"
    batch_size = 1   # important for Grad-CAM

    checkpoint_dir = "./checkpoints"
    model_file = "lung_swin_classifier_best.pth"

    output_dir = "./outputs/gradcam"

    device = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()
os.makedirs(cfg.output_dir, exist_ok=True)


# =========================================================
# MODEL (EXACT SAME AS TRAINING)
# =========================================================
class Head(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 2)
        )

    def forward(self, x):
        return self.net(x)


base_model = timm.create_model(cfg.model_name, pretrained=False, num_classes=0)
model = nn.Sequential(base_model, Head(base_model.num_features)).to(cfg.device)


# =========================================================
# LOAD MODEL
# =========================================================
checkpoint = torch.load(
    os.path.join(cfg.checkpoint_dir, cfg.model_file),
    map_location=cfg.device
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("Model loaded successfully")


# =========================================================
# DATA
# =========================================================
data_cfg = resolve_data_config({}, model=base_model)
data_cfg["crop_pct"] = 0.8

transform = create_transform(**data_cfg, is_training=False)

test_data = datasets.ImageFolder(cfg.test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=cfg.batch_size, shuffle=False)

class_names = test_data.classes

print(f"Total images: {len(test_data)}")


# =========================================================
# GRAD-CAM SETUP
# =========================================================
def reshape_transform(tensor):
    if len(tensor.shape) == 4:
        return tensor.permute(0, 3, 1, 2)

    grid = int(np.sqrt(tensor.size(1)))
    result = tensor.reshape(tensor.size(0), grid, grid, tensor.size(2))
    return result.permute(0, 3, 1, 2)


target_layers = [model[0].layers[-1].blocks[-1]]

cam = GradCAM(
    model=model,
    target_layers=target_layers,
    reshape_transform=reshape_transform
)


# =========================================================
# PROCESS IMAGES
# =========================================================
count = 0

for img_path, label in test_data.samples:

    # Load original image
    img_pil = Image.open(img_path).convert("RGB")
    img_resized = np.array(img_pil.resize((224, 224))) / 255.0

    input_tensor = transform(img_pil).unsqueeze(0).to(cfg.device)

    # Prediction
    with torch.no_grad():
        output = model(input_tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = torch.max(probs, dim=1)

    pred_class = class_names[pred.item()]
    confidence = conf.item() * 100

    # Grad-CAM
    targets = [ClassifierOutputTarget(pred.item())]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    # Mask (remove background)
    img_gray = cv2.cvtColor((img_resized * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(img_gray, 10, 1, cv2.THRESH_BINARY)
    mask = mask.astype(np.float32)

    cam_masked = grayscale_cam * mask

    # Smooth
    cam_smooth = cv2.GaussianBlur(cam_masked, (11, 11), 0)

    if cam_smooth.max() > 0:
        cam_smooth = (cam_smooth - cam_smooth.min()) / (cam_smooth.max() - cam_smooth.min())

    # Overlay
    cam_image = show_cam_on_image(img_resized.astype(np.float32), cam_smooth, use_rgb=True)

    # Save output
    class_folder = os.path.join(cfg.output_dir, pred_class)
    os.makedirs(class_folder, exist_ok=True)

    filename = os.path.basename(img_path).split('.')[0]

    save_path = os.path.join(
        class_folder,
        f"{filename}_pred-{pred_class}_conf-{confidence:.1f}.jpg"
    )

    cv2.imwrite(save_path, cv2.cvtColor(cam_image, cv2.COLOR_RGB2BGR))

    count += 1
    print(f"[{count}] Saved: {save_path}")


print("Grad-CAM generation completed")